In [2]:
from __future__ import annotations
from typing import List, Tuple, Dict, Optional
import torch
import esm
from Bio.PDB import PDBParser, Polypeptide
from Bio.Data import IUPACData

In [2]:
# 基础 3->1 字典（统一成大写键）
AA3_TO1 = {k.upper(): v for k, v in IUPACData.protein_letters_3to1.items()}

# 常见变体与别名补充
AA3_TO1.update({
    "MSE": "M",   # Selenomethionine -> treat as Met
    "SEC": "U",   # Selenocysteine
    "PYL": "O",   # Pyrrolysine
    # Histidine tautomer/PK forms seen in某些力场/建模软件
    "HSD": "H", "HSE": "H", "HSP": "H", "HIP": "H", "HID": "H", "HIE": "H",
    # Cys variants
    "CYM": "C", "CYX": "C",
})

def aa3_to1_safe(resname: str) -> str | None:
    """三字母转一字母；无法识别则返回 None 而不是抛错。"""
    name = resname.strip().upper()
    # 先走字典
    if name in AA3_TO1:
        return AA3_TO1[name]
    # 再尝试 biopython 自带的转换器
    try:
        return Polypeptide.three_to_one(name)
    except Exception:
        return None

def _is_std_aa(residue) -> bool:
    """是否为标准氨基酸：过滤水/配体/杂原子，仅接受能映射到 1-letter 的残基。"""
    het, resseq, icode = residue.id
    if het.strip() != "":           # 非空意味着 HETATM（含水 HOH、离子、配体等）
        return False
    aa1 = aa3_to1_safe(residue.get_resname())
    return aa1 is not None

def _res_uid(residue):
    """用 (resseq, icode) 唯一标识一个残基，兼容插入码。"""
    het, resseq, icode = residue.id
    return (int(resseq), (icode or " ").strip() or " ")

class ESM2PocketEmbedder:
    """
    - 从PDB提取单链序列 → ESM-2 残基embedding（不含BOS/EOS）
    - 根据 pocket PDB 中的残基集合，返回对应的embedding子集
    """
    def __init__(self,
                 model_name: str = "esm2_t33_650M_UR50D",
                 repr_layer: int = 33,
                 device: Optional[str] = None):
        self.model_name = model_name
        self.repr_layer = repr_layer
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        # 官方API：一次性加载模型与字母表
        self.model, self.alphabet = esm.pretrained.load_model_and_alphabet(model_name)
        self.model.eval().to(self.device)
        self.batch_converter = self.alphabet.get_batch_converter()

    def _get_chain(self, structure, chain_id: Optional[str]):
        model0 = next(structure.get_models())
        if chain_id is None:
            return next(model0.get_chains())  # 默认第一条链
        return model0[chain_id]

    def extract_residue_embeddings(self,
                                   pdb_path: str,
                                   chain_id: Optional[str] = None
                                   ) -> Tuple[List[Tuple[int, str, str]], torch.Tensor]:
        """
        返回：
          - res_list: [(resseq, icode, aa1), ...] 按序列顺序
          - emb: [L, H]  每个残基一个向量（不含BOS/EOS）
        """
        parser = PDBParser(QUIET=True)
        structure = parser.get_structure("protein", pdb_path)
        chain = self._get_chain(structure, chain_id)

        seq_chars: List[str] = []
        res_list: List[Tuple[int, str, str]] = []  # (resseq, icode, aa1)
        for res in chain.get_residues():
            if not _is_std_aa(res):
                continue
            key = _res_uid(res)
            aa3 = res.get_resname().upper()
            aa1 = aa3_to1_safe(aa3)
            if aa1 is None:
                print(f'not standard aa {aa3}')
                return  
            seq_chars.append(aa1)
            res_list.append((key[0], key[1], aa1))

        if len(seq_chars) == 0:
            raise ValueError("No standard amino acids found in the specified chain.")
        seq = "".join(seq_chars)

        data = [("protein", seq)]
        _, _, tokens = self.batch_converter(data)
        tokens = tokens.to(self.device)

        with torch.no_grad():
            out = self.model(tokens, repr_layers=[self.repr_layer], return_contacts=False)
        reps = out["representations"][self.repr_layer]  # [1, L+2, H]
        L = len(seq)
        per_res = reps[0, 1:1+L, :].detach().cpu()      # [L, H]
        return res_list, per_res

    def pocket_embeddings(self,
                          full_pdb_path: str,
                          pocket_pdb_path: str,
                          chain_id: Optional[str] = None
                          ) -> Tuple[List[Tuple[int, str, str]], torch.Tensor]:
        """
        基于 full PDB 的embedding，返回 pocket PDB 中残基的嵌入（按 pocket 文件残基顺序）。
        返回：
          - pocket_res_list: [(resseq, icode, aa1), ...]
          - pocket_emb: [K, H]
        说明：通过 (resseq, icode) 对齐，单链假设；若 pocket 含非标准残基会被跳过。
        """
        # 1) 全蛋白残基emb
        full_res_list, full_emb = self.extract_residue_embeddings(full_pdb_path, chain_id=chain_id)
        index_by_uid: Dict[Tuple[int, str], int] = {
            (r[0], r[1]): i for i, r in enumerate(full_res_list)
        }

        # 2) 读取口袋残基UID（按口袋PDB顺序）
        parser = PDBParser(QUIET=True)
        pocket_struct = parser.get_structure("pocket", pocket_pdb_path)
        pocket_chain = self._get_chain(pocket_struct, chain_id)

        pocket_keys: List[Tuple[int, str]] = []
        pocket_res_list: List[Tuple[int, str, str]] = []
        for res in pocket_chain.get_residues():
            if not _is_std_aa(res):
                continue
            uid = _res_uid(res)
            aa3 = res.get_resname().upper()
            aa1 = aa3_to1_safe(aa3)
            if aa1 is None:
                print(f'not standard aa {aa3}')
                return
            pocket_keys.append(uid)
            pocket_res_list.append((uid[0], uid[1], aa1))

        print(pocket_keys)
        print(pocket_res_list)
        
        if len(pocket_keys) == 0:
            raise ValueError("No standard amino acids found in pocket PDB.")

        # 3) 选择子集embedding
        idxs: List[int] = []
        missing: List[Tuple[int, str]] = []
        for uid in pocket_keys:
            if uid in index_by_uid:
                idxs.append(index_by_uid[uid])
            else:
                missing.append(uid)

        if missing:
            # 这里选择：忽略缺失并仅返回存在的；你也可以改成严格对齐并抛错
            # raise KeyError(f"Pocket residues not found in full protein embedding: {missing}")
            pass

        if len(idxs) == 0:
            raise RuntimeError("No overlapping pocket residues found in full protein embeddings.")

        idx_tensor = torch.tensor(idxs, dtype=torch.long)
        pocket_emb = full_emb.index_select(0, idx_tensor)  # [K, H]
        return pocket_res_list[:len(idxs)], pocket_emb

In [3]:
esm_2_embedder = ESM2PocketEmbedder(
    model_name="esm2_t33_650M_UR50D",
    repr_layer=33,
    device="cuda" if torch.cuda.is_available() else "cpu"
)


In [4]:
full_res_list, full_emb = esm_2_embedder.extract_residue_embeddings('/home/yang2531/Documents/Project/GNN_VAE_draft/data/small_frag/high/target_CHEMBL202/CHEMBL22/protein.pdb', chain_id=None)
print(len(full_res_list), full_emb.shape)  # e.g. (L, [L, 1280])

186 torch.Size([186, 1280])


In [5]:
pocket_res_list, pocket_emb = esm_2_embedder.pocket_embeddings('/home/yang2531/Documents/Project/GNN_VAE_draft/data/small_frag/high/target_CHEMBL202/CHEMBL22/protein.pdb', 
                                                               "/home/yang2531/Documents/Project/GNN_VAE_draft/data/small_frag/high/target_CHEMBL202/CHEMBL22/protein_6A.pdb", 
                                                               chain_id=None)
print(len(pocket_res_list), pocket_emb.shape)  # K, [K, 1280]

[(7, ' '), (8, ' '), (9, ' '), (22, ' '), (24, ' '), (30, ' '), (31, ' '), (33, ' '), (34, ' '), (35, ' '), (50, ' '), (52, ' '), (56, ' '), (57, ' '), (59, ' '), (60, ' '), (61, ' '), (64, ' '), (65, ' '), (67, ' '), (70, ' '), (114, ' '), (115, ' '), (116, ' '), (121, ' '), (134, ' '), (135, ' '), (136, ' '), (179, ' ')]
[(7, ' ', 'I'), (8, ' ', 'V'), (9, ' ', 'A'), (22, ' ', 'L'), (24, ' ', 'W'), (30, ' ', 'E'), (31, ' ', 'F'), (33, ' ', 'Y'), (34, ' ', 'F'), (35, ' ', 'K'), (50, ' ', 'V'), (52, ' ', 'M'), (56, ' ', 'T'), (57, ' ', 'W'), (59, ' ', 'S'), (60, ' ', 'I'), (61, ' ', 'P'), (64, ' ', 'F'), (65, ' ', 'R'), (67, ' ', 'L'), (70, ' ', 'R'), (114, ' ', 'I'), (115, ' ', 'V'), (116, ' ', 'G'), (121, ' ', 'Y'), (134, ' ', 'F'), (135, ' ', 'V'), (136, ' ', 'T'), (179, ' ', 'F')]
29 torch.Size([29, 1280])


In [6]:
# 3) 若需要一个口袋定长向量（mean/attention pooling）
# 我们可以用attention pooling,但一切从简，先跑完再说。
pocket_vec = pocket_emb.mean(dim=0)  # [H]
print(pocket_vec)
pocket_vec.unsqueeze(0).shape

tensor([-0.0189, -0.1878, -0.0189,  ..., -0.2960,  0.0295,  0.1792])


torch.Size([1, 1280])